In [ ]:
import numpy as np

from waffles.data_classes.WaveformSet import WaveformSet
from waffles.data_classes.EasyWaveformCreator import EasyWaveformCreator
from waffles.data_classes.ChannelWsGrid import ChannelWsGrid
from waffles.np02_utils.PlotUtils import matplotlib_plot_WaveformSetGrid, dict_module_to_uniqch, dict_uniqch_to_module, runBasicWfAnaNP02, wfset_remove_bad_baselines
from waffles.np02_utils.AutoMap import ordered_channels_cathode, ordered_modules_membrane, getModuleName, dict_endpoints_channels_list
from waffles.np02_utils.AutoMap import ordered_modules_cathode, ordered_modules_membrane, ordered_modules_pmt, getModuleName
from waffles.np02_utils.load_utils import open_processed, remove_extra_channels_membrane
from waffles.utils.fft.fftutils import FFTWaffles
from waffles.utils.numerical_utils import average_wf_ch

In [ ]:
runs = [42530, 42945]
runs = [43015, 42422]
# runs = [37286, 37287]
labels_list = [ "UPS  ON", "UPS OFF" ]
dettype = "membrane"
dettype = "cathode"
# dettype = "pmt"
nwaveforms = None


labels = { run: f"{lb} - {run}" for run, lb in zip(runs, labels_list) }
datadir = f"/eos/experiment/neutplatform/protodune/experiments/ProtoDUNE-VD/commissioning/"
endpoint = 106 if dettype == "cathode" else 107 if dettype== "membrane" else 110

listmods = ordered_modules_cathode if endpoint == 106 else ordered_modules_membrane if endpoint == 107 else ordered_modules_pmt
n = len(listmods)
group1 = listmods[:n//4]
group2 = listmods[n//4:n//2]
group3 = listmods[n//2:2*(n//3+1)]
group4 = listmods[2*(n//3+1):]
    
groupLow = group1+group2
groupHig = group3+group4
groupall = group1+group2+group3+group4

In [ ]:
def generate_ffts(run, dettype, datadir, endpoint, nwavefroms):
    wfset_full = open_processed(run, dettype, datadir, nwaveforms=nwaveforms, mergefiles=True, file_slice=slice(None,1))
    wfset_full = WaveformSet.from_filtered_WaveformSet(wfset_full, remove_extra_channels_membrane)
    runBasicWfAnaNP02(wfset_full, threshold=100, onlyoptimal=True, configyaml="")
    wfset_full = wfset_remove_bad_baselines(wfset_full)
    wfsetch = ChannelWsGrid.clusterize_waveform_set(wfset_full)[endpoint]
    wfsetch = { k: v for k, v in sorted(wfsetch.items(), key=lambda x: getModuleName(endpoint, x[0])) }
    wfft = FFTWaffles()
    ffts = { run: {} }
    for ch, wfs in wfsetch.items():
        print( getModuleName(endpoint, ch), len(wfs.waveforms) )
        # avg = average_wf_ch(wfs)
        listffts = [ wfft.getFFT( wf.adcs - wf.analyses['std'].result['baseline']) for wf in wfs.waveforms ] 
        ffts[list(wfs.runs)[0]][ch] = np.mean(listffts, axis=0)
    return ffts

In [ ]:
ffts = {} 
for run in runs:
    dictffts = generate_ffts(run, dettype, datadir, endpoint, nwaveforms)
    ffts.update(dictffts)

In [ ]:
wfset_grid = EasyWaveformCreator.create_WaveformSet_dictEndpointCh(dict_endpoints_channels_list)

In [ ]:
import matplotlib.pyplot as plt
def plot_fft(wfset:WaveformSet, ffts:dict, labels:dict = {}):
    wfft = FFTWaffles()
    ch = wfset.waveforms[0].channel
    ep = wfset.waveforms[0].endpoint
    mname = getModuleName(ep,ch)
    for run, chfft in ffts.items():
        if ch not in chfft:
            continue
        x=wfft.getXFreq(62.5, 2*(len(chfft[ch])-1))
        yfft = 20 * np.log10(np.abs(chfft[ch]) / 2**14)
        plt.plot(x, yfft, label=labels.get(run, run))
        # plt.yscale('log')
        plt.ylabel("FFT amplitude [dBFS]")
        plt.xlabel("Frequency [MHz]")
        legtitle= f"{mname}: {ep}-{ch}"
        plt.legend(title=legtitle)
        plt.xscale('log')
fig = matplotlib_plot_WaveformSetGrid(wfset_grid, groupall, plot_function=plot_fft, func_params={'ffts': ffts, 'labels': labels}, figsize=(24,18))
plt.tight_layout()
runsinfo = '_'.join(map(str,runs))
plt.savefig(f"ffts_{dettype}_{runsinfo}")